In [1]:
import pandas as pd
import numpy as np

## Preparation

In [2]:
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/3_Player_Data_Generation/match_data_50_tourns_modified.csv')
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_played_3_years,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,446,242,32,17,32,17,5,0,0.0,1.000000
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,23,4,245,118,808,430,2,5,1.0,0.285714
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,404,215,101,49,336,147,5,3,0.0,0.625000
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,534,282,91,35,235,97,5,2,0.0,0.714286
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,167,61,281,170,1112,649,2,5,1.0,0.285714


In [3]:
#Add more features
data['p1_frames_win_rate'] = data['p1_frames_won']/ data['p1_frames_played']
data['p2_frames_win_rate'] = data['p2_frames_won']/ data['p2_frames_played']

data['p1_matches_win_rate'] = data['p1_matches_won']/ data['p1_matches_played']
data['p2_matches_win_rate'] = data['p2_matches_won']/ data['p2_matches_played']

#p1_matches_win_rate and p2_matches_win_rate both have missing values
#We will fill them with 0.5
data.fillna(0.5, inplace = True)

#Drop players' names from the data since we won't consider strings as our features for modeling
data = data.drop(['player1', 'player2'], axis = 1)

In [4]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle=False)


In [5]:
#Create predictors and targets for training and cross-validation set
y_train = data_train['match_result']
y_test = data_test['match_result']

#To get the predictor, we exclude players' names (Strings) and match results.
X_train = data_train.drop(['match_result', 'win_percentage', 'score1', 'score2'], axis = 1)
X_test = data_test.drop(['match_result', 'win_percentage','score1', 'score2'], axis = 1)

## Performance Metrics
Because the two classes in the target are symmetric (swapping player1 and player2 will exchange positive and negative but still represents the same match), we won't consider metrics such as presision, specificity and sensitivity since they are the same as accuracy score. We will only consider accuracy score.

In [6]:
#Import metrics
from sklearn.metrics import accuracy_score

In [7]:
#Create dictionaries to store the metrics.
accuracy_scores = {}

#Create a list to store all the models we consider.
models = {}

In [8]:
def print_avg_cv_metrics(model, model_name):
    """
    Given a model, computes and stores accuracy scores on the test set.
    """

    #Create an empty array to store the scores.
    print('Currently working on ' + model_name + '.')

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    score = accuracy_score(y_test, y_pred)

    print('The accuracy score of ' + model_name + ' is:', score)
    
    #Record the scores
    accuracy_scores[model_name] = score

    models[model_name] = model


## Model 3: SVC

In [9]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


In [10]:
scaler = StandardScaler()
svc = SVC()

svc = Pipeline([('scale', scaler),
                            ('svc', svc)])

param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__kernel": ['linear', 'rbf'],
    "svc__gamma": np.logspace(-2, 2, 5)
}


grid_search3 = GridSearchCV(svc,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search3.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('svc', SVC())]),
             param_grid={'svc__C': [0.1, 1, 10, 100],
                         'svc__gamma': array([1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02]),
                         'svc__kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [11]:
print(grid_search3.best_params_)
print(grid_search3.best_score_)

{'svc__C': 10, 'svc__gamma': np.float64(0.01), 'svc__kernel': 'rbf'}
0.6846214653054373


In [12]:
model3 = grid_search3.best_estimator_
print_avg_cv_metrics(model3, 'SVC')

Currently working on SVC.
The accuracy score of SVC is: 0.6284584980237155


## Model 4: SVC with PCA

In [13]:
from sklearn.decomposition import PCA

In [14]:
scaler = StandardScaler()
svc = SVC()
pca = PCA(n_components=10)

svc_with_pca = Pipeline([('scale', scaler),
                ('pca', pca),
                ('svc', svc)])

param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__kernel": ['linear', 'rbf'],
    "svc__gamma": np.logspace(-2, 2, 5)
}


grid_search4 = GridSearchCV(svc_with_pca,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search4.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('pca', PCA(n_components=10)),
                                       ('svc', SVC())]),
             param_grid={'svc__C': [0.1, 1, 10, 100],
                         'svc__gamma': array([1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02]),
                         'svc__kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [15]:
print(grid_search4.best_params_)
print(grid_search4.best_score_)

{'svc__C': 1, 'svc__gamma': np.float64(0.01), 'svc__kernel': 'rbf'}
0.6833862869874407


In [16]:
model4 = grid_search4.best_estimator_
print_avg_cv_metrics(model4, 'SVC with pca')

Currently working on SVC with pca.
The accuracy score of SVC with pca is: 0.6304347826086957


## Model 5: Linear Discriminant Analysis

In [17]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [18]:
lda = LinearDiscriminantAnalysis()

param_grid = {
    "solver": ['lsqr', 'eigen'],
    "shrinkage": ['auto', 0, 0.01, 0.1, 1]
}

grid_search5 = GridSearchCV(lda,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)

grid_search5.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LinearDiscriminantAnalysis(),
             param_grid={'shrinkage': ['auto', 0, 0.01, 0.1, 1],
                         'solver': ['lsqr', 'eigen']},
             scoring='accuracy')

In [19]:
print(grid_search5.best_params_)
print(grid_search5.best_score_)

{'shrinkage': 'auto', 'solver': 'lsqr'}
0.6843766881838576


In [20]:
model5 = grid_search5.best_estimator_
print_avg_cv_metrics(model5, 'SVC with pca')

Currently working on SVC with pca.
The accuracy score of SVC with pca is: 0.6413043478260869


## Model 6: XGBoost

In [21]:
import xgboost as xgb

In [22]:
XGB = xgb.XGBClassifier()

param_grid = {
    "max_depth": [3,4,5,6, 7, 8],
    "eta": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200, 300, 400, 500, 600, 700], 
    "subsample": [0.5, 1]
}

grid_search6 = GridSearchCV(XGB,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)

grid_search6.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None,...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None,
                                     random_state=None, ...),
             param_grid={'eta': [0.01, 0.05, 0.1],
                         'max_depth': [3, 4, 5, 6, 7, 8],
                         'n_estimators': [100, 200, 300, 400, 500, 600, 700],
                         'subsample': [0.5, 1]},
             scoring='accuracy')

In [23]:
print(grid_search6.best_params_)
print(grid_search6.best_score_)

{'eta': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.5}
0.6903059714019747


In [24]:
model6 = grid_search6.best_estimator_
print_avg_cv_metrics(model6, 'XGBoost')

Currently working on XGBoost.
The accuracy score of XGBoost is: 0.6640316205533597


In [25]:
print(accuracy_scores)

{'SVC': 0.6284584980237155, 'SVC with pca': 0.6413043478260869, 'XGBoost': 0.6640316205533597}
